# Attention weight density analysis
## Treating seeds as subjects

In [ ]:
from lstnn.compare_rdms import get_transformer_weights
from lstnn.dataset import get_dataset
import numpy as np
import matplotlib.pyplot as plt
import pingouin as pg
import seaborn as sns
import pandas as pd

data_dir = "../processed_data/"
rdm_method_ann = "euclidean"
pe = "2dpe"
epoch = 4000
# load test puzzles as a torch ds
LST_puzzle_ds = get_dataset(f"{data_dir}puzzle_data_original.csv")

## Target column vs. all other columns

In [ ]:
from lstnn.verify_LST import verify_LST
from lstnn.dataset import get_dataset
import numpy as np
import matplotlib.pyplot as plt
import pingouin as pg
import seaborn as sns
import pandas as pd


# load test puzzles as a torch ds
LST_puzzle_ds = get_dataset(f"{data_dir}puzzle_data_original.csv")

# get ANN weights
ann_weights = get_transformer_weights(LST_puzzle_ds, pe, epoch, data_dir=data_dir)
attn = ann_weights["attn"].copy()
n_seeds = attn.shape[0]
n_layers = attn.shape[1]

In [ ]:
# This cell processes attention weights to analyze mean key attention (MKA) values
# for target vs. non-target positions across different puzzle complexity conditions.
# It loops through each seed and puzzle, extracts attention data, averages across layers,
# reshapes to a 16x16 (token by token) attention matrix, computes column means, identifies the target
# (question mark) position, and compares target attention vs. average non-target attention.
# Results are stored in a dataframe with seed, puzzle ID, target status, attention value,
# and puzzle condition (1/2/3) for later statistical analysis.
results = []
for s in range(n_seeds):
    for p in range(108):

        # ATTN data
        data = attn[s, :, p, :]

         # average across layers
        data = np.mean(data, axis=0) 

        # reshape to attention matrix
        data = data.reshape(16, 16)
        mean_key_vals = np.mean(data, axis=0)
        
        # find the question mark for this puzzle
        target_index = np.where(LST_puzzle_ds.print_puzzle(p)[0].reshape(-1) == 5)[0][0]
        
        # get the mean of the target col
        target_mean = mean_key_vals[target_index]

        # get all other means
        mask = np.zeros(len(mean_key_vals), dtype=bool)
        mask[target_index] = True
        other_means = np.mean(mean_key_vals[~mask])

        # define condition
        if p < 36:
            condition = 1
        elif 36 <= p < 72:
            condition = 2
        else:
            condition = 3

        _df = pd.DataFrame()
        _df["seed"] = [s]
        _df["puzzle"] = p
        _df["target"] = True
        _df["value"] = target_mean
        _df["condition"] = condition
        results.append(_df)

        _df = pd.DataFrame()
        _df["seed"] = [s]
        _df["puzzle"] = p
        _df["target"] = False
        _df["value"] = other_means
        _df["condition"] = condition
        results.append(_df)

mkw_results = pd.concat(results)
mkw_results.head()

In [ ]:
import pingouin as pg
data = mkw_results.groupby(["seed","target"]).mean(numeric_only=False).reset_index()
display(data.head(3))
print(data.shape)
sns.catplot(data=data, y="value", x="target", dodge=True, kind="box")
display(pg.ttest(x=data.loc[data.target==True, "value"].values,
         y=data.loc[data.target==False, "value"].values, paired=True))


# Contrast conditions

In [ ]:
data = mkw_results.loc[mkw_results.target==True].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
sns.catplot(data=data, y="value", x="condition", height=3, kind="point", sharey=False)
plt.show()

data = mkw_results.loc[mkw_results.target==False].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
sns.catplot(data=data, y="value", x="condition", height=3, kind="point", sharey=False)
plt.show()

### Within-subject variance is important for this effect:

In [ ]:
def cousineau_morey_ci(data):
    n_subjects = data.shape[0]

    # Normalize the data by subtracting the subject mean and adding the grand mean
    subject_means = np.mean(data, axis=1)
    grand_mean = np.mean(data)
    normalized_data = (data - subject_means[:, np.newaxis]) + grand_mean

    # CI = Mean +/- t(1-alpha/2, n-1) * SD / sqrt(n) * sqrt(c / (c-1))
    critical_value = 1.96
    m = np.mean(normalized_data, axis=0)
    sem = np.std(normalized_data, axis=0) / np.sqrt(n_subjects)
    correction = np.sqrt(data.shape[1] / (data.shape[1]-1))

    # Calculate the confidence intervals
    ci_lower = m - (critical_value * sem * correction)
    ci_upper = m + (critical_value * sem * correction)

    return ci_lower, ci_upper

data = mkw_results.loc[mkw_results.target==True].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
y = "value"
ax = sns.pointplot(data=data, x="condition", y=y, errorbar=None)

# plot within-subject CIs
ci_data = np.vstack((data.loc[data.condition == 1, y].values,
                    data.loc[data.condition == 2, y].values,
                    data.loc[data.condition == 3, y].values)).T
cis_upper, cis_lower = cousineau_morey_ci(ci_data)
for j in range(3):
    ax.plot([j, j], [cis_upper[j], cis_lower[j]])
    
sns.despine(ax=ax)
plt.show()

data = mkw_results.loc[mkw_results.target==False].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
y = "value"
ax = sns.pointplot(data=data, x="condition", y=y, errorbar=None)

# plot within-subject CIs
ci_data = np.vstack((data.loc[data.condition == 1, y].values,
                    data.loc[data.condition == 2, y].values,
                    data.loc[data.condition == 3, y].values)).T
cis_upper, cis_lower = cousineau_morey_ci(ci_data)
for j in range(3):
    ax.plot([j, j], [cis_upper[j], cis_lower[j]])
    
sns.despine(ax=ax)
plt.show()

In [ ]:
data = mkw_results.loc[mkw_results.target==True].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
display(pg.rm_anova(data=data, dv="value", within="condition", subject="seed"))
display(pg.friedman(data=data, dv="value", within="condition", subject="seed"))

data = mkw_results.loc[mkw_results.target==False].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
display(pg.rm_anova(data=data, dv="value", within="condition", subject="seed"))
display(pg.friedman(data=data, dv="value", within="condition", subject="seed"))

In [ ]:
# ttests
data = mkw_results.loc[mkw_results.target==False].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
pvals = []

print("1 v. 2")
y="value"
res = pg.ttest(x=data.loc[data.condition==1, y].values,
         y=data.loc[data.condition==2, y].values, paired=True)
display(res)
pvals.append(res["p-val"].values[0])

print("2 v. 3")
res = pg.ttest(x=data.loc[data.condition==2, y].values,
          y=data.loc[data.condition==3, y].values, paired=True)
display(res)
pvals.append(res["p-val"].values[0])

print("1 v. 3")
res = pg.ttest(x=data.loc[data.condition==1, y].values,
         y=data.loc[data.condition==3, y].values, paired=True)
display(res)
pvals.append(res["p-val"].values[0])

print(pg.multicomp(pvals, method="holm"))

# Informative vs. non-informative cells

In [ ]:
# load test puzzles as a torch ds
LST_puzzle_ds = get_dataset(f"{data_dir}puzzle_data_original.csv")

inf_results = []
n_seeds = attn.shape[0]
n_layers = attn.shape[1]
print(LST_puzzle_ds.print_puzzle(3))
for s in range(n_seeds):
    for p in range(36):

        # ATTN data
        data = attn[s, :, p, :]
        data = np.mean(data, axis=0)

        # reshape to attention matrix
        data = data.reshape(16, 16)
        mean_key_vals = np.mean(data, axis=0)

        # find the question mark for this puzzle
        target_index = np.where(LST_puzzle_ds.print_puzzle(p)[0].reshape(-1) == 5)[0][0]

        # get strat:
        strat = verify_LST(LST_puzzle_ds.print_puzzle(p)[0].numpy())

        # find the row of interest
        # these should be sequential indicies - as the reshape doesn't mix up rows
        row_of_interest, col_of_interest = np.where(LST_puzzle_ds.print_puzzle(p)[0] == 5)
        row_index = np.zeros((4,4))
        row_index[row_of_interest[0], :] = True

        # find the cols of interest - these are not sequential
        col_index = np.zeros((4,4))
        col_index[:, col_of_interest[0]] = True

        # take only values in the row (remove the target)
        idx = np.where(row_index.reshape(-1))[0]
        idx = np.delete(idx, np.where(idx==target_index))
        row_values  = mean_key_vals[idx]

        #take only values in the col (remove the target)
        idx = np.where(col_index.reshape(-1))[0]
        idx = np.delete(idx, np.where(idx==target_index))
        col_values  = mean_key_vals[idx]

        # take all other values

        if strat[2] == "Binary-row":
            info_cells = np.mean(row_values)
            #non_cells = np.mean(col_values)
            inv_index = (row_index * -1 )+ 1
            idx = np.where(inv_index.reshape(-1))[0]
            values  = mean_key_vals[idx]
            non_cells = np.mean(values)
            
        elif strat[2] == "Binary-col":
            #non_cells = np.mean(row_values)
            info_cells = np.mean(col_values)
            inv_index = (col_index * -1 )+ 1
            idx = np.where(inv_index.reshape(-1))[0]
            values  = mean_key_vals[idx]
            non_cells = np.mean(values)

        # save 
        _df = pd.DataFrame()
        _df["seed"] = [s]
        _df["puzzle"] = p
        _df["cond"] = 1
        _df["cell"] = "informative"
        _df["value"] = info_cells
        inf_results.append(_df)

        # save
        _df = pd.DataFrame()
        _df["seed"] = [s]
        _df["puzzle"] = p
        _df["cond"] = 1
        _df["cell"] = "uninformative"
        _df["value"] = non_cells
        inf_results.append(_df)
inf_results = pd.concat(inf_results)
inf_results.head()

In [ ]:
x = inf_results.groupby(["seed", "cell"]).mean(numeric_only=True).reset_index()
sns.boxplot(data=x, y="value", x="cell")
x.head()

In [ ]:
x = inf_results.groupby(["seed", "cell"]).mean(numeric_only=True).reset_index()
x.head()
display(pg.ttest(x=x.loc[x.cell=="informative", "value"].values,
                 y=x.loc[x.cell=="uninformative", "value"].values, paired=True))

# Publication plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
#88mm = 3.46 inch

plt.rcParams['svg.fonttype'] = 'none'
pal = [(0.918, 0.451, 0.553),
       (0.537, 0.671, 0.890)]

# create figure
fig, axs = plt.subplot_mosaic("""
                              ABCC
                              """,
                              figsize=(3.96, 1.8),
                              constrained_layout=True)

ax = axs["A"]
y="value"
data = mkw_results.groupby(["seed","target"]).mean(numeric_only=False).reset_index()
sns.pointplot(data=data, x="target", y=y, errorbar=None, color="k", ax=ax, linewidth=1, markersize=5)

# plot within-subject CIs
ci_data = np.vstack((data.loc[data.target == False, y].values,
                    data.loc[data.target == True, y].values)).T
cis_upper, cis_lower = cousineau_morey_ci(ci_data)
for j in range(2):
    ax.plot([j, j], [cis_upper[j], cis_lower[j]], color="k")

ax.tick_params(axis='both', which='major', labelsize=8)
ax.set_ylabel("MKA weight", fontsize=9, labelpad=0)
ax.set_xticklabels(labels=["Non-Target", "Target"], fontsize=9)
ax.set_xlabel("")
ax.set_xlim([-0.2, 1.2])
#ax.set_ylim([0, 0.25])
#ax.set_yticks([0.0, 0.125, 0.25])
sns.despine(ax=ax)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

ax.yaxis.get_offset_text().set_fontsize(7)
offset = ax.yaxis.get_offset_text()
offset.set_x(-0.15)   # move left/right (axes fraction)

ax = axs["B"]

data = inf_results.groupby(["seed", "cell"]).mean(numeric_only=True).reset_index()
sns.pointplot(data=data, x="cell", y="value", errorbar=None, 
              order=["uninformative", "informative"], ax=ax, color="k", linewidth=1, markersize=5, legend=False)

# plot within-subject CIs
ci_data = np.vstack((data.loc[data.cell == "uninformative", y].values,
                    data.loc[data.cell == "informative", y].values)).T
cis_upper, cis_lower = cousineau_morey_ci(ci_data)
for j in range(2):
    ax.plot([j, j], [cis_upper[j], cis_lower[j]], color="k")

ax.tick_params(axis='both', which='major', labelsize=8)
ax.set_ylabel("MKA weight", fontsize=9, labelpad=0)
ax.set_xticklabels(labels=["Non-critical", "Critical"], fontsize=9)
ax.set_xlabel("")
ax.set_xlim([-0.2, 1.2])
# ax.set_ylim([0.18, 0.3])
# ax.set_yticks([0.20, 0.25, 0.3])
sns.despine(ax=ax)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.yaxis.get_offset_text().set_fontsize(7)
offset = ax.yaxis.get_offset_text()
offset.set_x(-0.15)   # move left/right (axes fraction)

ax = axs["C"]

data = mkw_results.loc[mkw_results.target==False].copy()
data = data.groupby(["seed", "condition"]).mean(numeric_only=True).reset_index()
y = "value"
sns.pointplot(data=data, x="condition", y=y, errorbar=None, ax=ax, color="k", linewidth=1, markersize=5, legend=False)

# plot within-subject CIs
ci_data = np.vstack((data.loc[data.condition == 1, y].values,
                    data.loc[data.condition == 2, y].values,
                    data.loc[data.condition == 3, y].values)).T
cis_upper, cis_lower = cousineau_morey_ci(ci_data)
for j in range(3):
    ax.plot([j, j], [cis_upper[j], cis_lower[j]], color="k")

ax.tick_params(axis='both', which='major', labelsize=8)
ax.set_ylabel("Non-target MKA weight", fontsize=9, labelpad=0)
ax.set_yticks([0.0515, 0.0520, 0.0525])
ax.set_xticklabels(labels=["1-vec", "2-vec", "3-vec"], fontsize=9)
ax.set_xlabel("Complexity")
ax.set_xlim([-0.2, 2.2])
sns.despine(ax=ax)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.yaxis.get_offset_text().set_fontsize(7)
offset = ax.yaxis.get_offset_text()
offset.set_x(-0.15)   # move left/right (axes fraction)
plt.show()